# Manga Image Translator - Google Colab

Translate manga/images into your desired language using AI.

**⚠️ Make sure to select a GPU runtime before running:**
`Runtime` → `Change runtime type` → Select `T4 GPU` (or better)

In [ ]:
# Clone the repository (fork with Load Text / Export OCR features)
!git clone https://github.com/mugnimaestra/manga-image-translator
!cd manga-image-translator && git checkout feature/web-ui-load-export-text
%cd /content/manga-image-translator/

# ==============================================================================
# 📦 Install Dependencies
# ==============================================================================
# Strategy: Install CUDA PyTorch FIRST (with all nvidia deps intact),
# THEN install project requirements. pip sees torch as "already satisfied"
# and won't overwrite it with CPU-only torch.
#
# Why not --force-reinstall --no-deps? It skips ALL 13 nvidia sub-packages
# (cublas, cudnn, cusparse, cusparselt, nvjitlink, etc.) that torch dynamically
# links against. Missing ANY ONE of them causes ImportError at 'import torch'.
# ==============================================================================

# Step 1: Install CUDA-enabled PyTorch FIRST with all nvidia dependencies
# This must happen BEFORE requirements.txt to prevent CPU torch from being installed
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu124

# Step 2: Install project requirements
# pip sees "torch" already installed (cu124 build) → "Requirement already satisfied"
# → no overwrite, all nvidia packages survive intact
!pip install -r requirements.txt

print('\n✅ Installation complete!')

In [ ]:
# Verify CUDA is available (should print True)
import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('❌ CUDA not available! Make sure you selected a GPU runtime.')
    print('Go to: Runtime -> Change runtime type -> Select T4 GPU')

## (Optional) Set translator API keys

If you want to use cloud-based translators (e.g., GPT, DeepL, Gemini), set the corresponding
environment variables below. Skip this cell if you only need the default translator.

In [ ]:
# Uncomment and fill in the API keys for the translators you want to use
import os

# OpenAI / ChatGPT
# os.environ['OPENAI_API_KEY'] = 'your-key-here'
# os.environ['OPENAI_MODEL'] = 'gpt-4o-mini'

# DeepL
# os.environ['DEEPL_AUTH_KEY'] = 'your-key-here'

# Google Gemini
# os.environ['GOOGLE_GEMINI_API_KEY'] = 'your-key-here'

## Option A: Web UI Mode

Run the web server and use the browser-based interface. Supports Load Text and Export OCR Text features.

In [ ]:
# Start the web server with Colab proxy
from google.colab.output import eval_js

port = 8000
url = eval_js(f"google.colab.kernel.proxyPort({port})")
print(f'Open this link to use manga-image-translator → {url}')

!python server/main.py --host 0.0.0.0 --port {port} --use-gpu --verbose

## Option B: CLI Batch Mode - OCR Export → Manual Edit → Load Text

A 3-step workflow for manual translation control:
1. **Export OCR** — detect text and save raw OCR as JSON
2. **Edit** — manually translate/fix the text in the JSON file
3. **Load + Render** — load your translations and render the final image

This is optimized for T4 GPU (16 GB VRAM).

In [ ]:
# Upload your manga images
import os
from google.colab import files

upload_dir = '/content/manga_input'
os.makedirs(upload_dir, exist_ok=True)

print('Upload your manga images:')
uploaded = files.upload()
for filename in uploaded:
    dest = os.path.join(upload_dir, filename)
    with open(dest, 'wb') as f:
        f.write(uploaded[filename])
    print(f'  Saved: {dest}')

print(f'\n✅ {len(uploaded)} image(s) uploaded to {upload_dir}')

In [ ]:
# Create T4-optimized config files
import json

# Config for Step 1: Export OCR only (lightweight — no inpainter loaded)
export_config = {
    "detector": {
        "detector": "default",
        "detection_size": 1536,
        "text_threshold": 0.5,
        "box_threshold": 0.7,
        "unclip_ratio": 2.3
    },
    "ocr": {
        "ocr": "48px",
        "min_text_length": 0
    },
    "translator": {
        "translator": "none",
        "target_lang": "ENG"
    },
    "inpainter": {
        "inpainter": "none"
    },
    "upscale": {
        "upscaler": "esrgan",
        "upscale_ratio": None
    },
    "colorizer": {
        "colorizer": "none"
    }
}

# Config for Step 3: Load text + inpaint + render
render_config = {
    "detector": {
        "detector": "default",
        "detection_size": 1536,
        "text_threshold": 0.5,
        "box_threshold": 0.7,
        "unclip_ratio": 2.3
    },
    "ocr": {
        "ocr": "48px",
        "min_text_length": 0
    },
    "translator": {
        "translator": "none",
        "target_lang": "ENG"
    },
    "inpainter": {
        "inpainter": "lama_large",
        "inpainting_size": 1024,
        "inpainting_precision": "bf16"
    },
    "render": {
        "renderer": "manga2eng",
        "alignment": "auto",
        "direction": "auto",
        "font_size_offset": 0,
        "no_hyphenation": True
    },
    "upscale": {
        "upscaler": "esrgan",
        "upscale_ratio": None
    },
    "colorizer": {
        "colorizer": "none"
    },
    "mask_dilation_offset": 20,
    "kernel_size": 3
}

with open('/content/t4_export_ocr.json', 'w') as f:
    json.dump(export_config, f, indent=2)

with open('/content/t4_load_render.json', 'w') as f:
    json.dump(render_config, f, indent=2)

print('✅ T4-optimized config files created:')
print('  /content/t4_export_ocr.json   (Step 1: ~2-3 GB VRAM)')
print('  /content/t4_load_render.json  (Step 3: ~6-8 GB VRAM)')

### Step 1: Export Raw OCR Text

Runs detection + OCR → saves detected text as JSON → exits.
No translation or inpainting model is loaded.

In [ ]:
%cd /content/manga-image-translator

# Export raw OCR text (no translation, no inpainting loaded)
!python -m manga_translator local \
    --save-text \
    --use-gpu \
    -i /content/manga_input/ \
    --config-file /content/t4_export_ocr.json

In [ ]:
# Find and display the exported OCR text files
import glob, json

ocr_files = glob.glob('result/*_translations.txt') + glob.glob('results/*_translations.txt')
if not ocr_files:
    print('⚠️ No OCR text files found. Make sure Step 1 ran successfully.')
else:
    for f in ocr_files:
        print(f'📄 {f}')
        with open(f, 'r') as fh:
            data = json.load(fh)
        for i, text in enumerate(data):
            print(f'  [{i}] {text}')
        print()

    # Download for editing
    from google.colab import files
    for f in ocr_files:
        files.download(f)
    print('✅ Downloaded! Edit the JSON file and re-upload in the next step.')

### Step 2: Upload Edited Translations

Edit the downloaded JSON file — replace the OCR text with your translations.

The format is a simple JSON array:
```json
["Translation for bubble 1", "Translation for bubble 2", "..."]
```

Then upload it back:

In [ ]:
# Upload your edited translations file
import shutil, glob
from google.colab import files

print('Upload your edited _translations.txt file:')
uploaded = files.upload()

# Replace the original file with the edited version
existing_files = glob.glob('result/*_translations.txt') + glob.glob('results/*_translations.txt')
for filename in uploaded:
    if existing_files:
        dest = existing_files[0]  # Replace the first match
    else:
        dest = f'result/{filename}'
    with open(dest, 'wb') as f:
        f.write(uploaded[filename])
    print(f'✅ Saved edited translations to: {dest}')

### Step 3: Load Translations + Inpaint + Render

Loads your edited translations, runs inpainting to remove original text, and renders your translations onto the image.

In [ ]:
%cd /content/manga-image-translator

# Load edited translations + inpaint + render
!python -m manga_translator local \
    --load-text \
    --use-gpu \
    -i /content/manga_input/ \
    -o /content/manga_output/ \
    --config-file /content/t4_load_render.json

In [ ]:
# Display and download results
import glob
from IPython.display import display, Image
from google.colab import files

output_files = glob.glob('/content/manga_output/**/*.*', recursive=True)
output_files = [f for f in output_files if f.lower().endswith(('.png', '.jpg', '.jpeg', '.webp'))]

if not output_files:
    print('⚠️ No output images found.')
else:
    for f in output_files:
        print(f'🖼️ {f}')
        display(Image(filename=f, width=400))
        files.download(f)
    print(f'\n✅ {len(output_files)} image(s) downloaded!')